<a href="https://colab.research.google.com/github/DanielBerns/python_para_universitarios/blob/main/curso_c/notebooks_introduccion/07_sqlite_pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 7: Integración de SQL con Pandas

Las bases de datos relacionales son el estándar industrial para almacenar grandes volúmenes de datos estructurados. En esta sesión aprenderás a conectar Python con una base de datos local **SQLite** y extraer información directamente a DataFrames de **Pandas** usando consultas SQL SQL de selección (`SELECT`).

## 1. Conexión a SQLite y Carga con Pandas
El módulo integrado `sqlite3` permite crear una conexión con la base de datos. Una vez establecida la conexión, Pandas nos facilita la vida con la función `pd.read_sql_query()`, la cual ejecuta la consulta y nos devuelve un DataFrame estructurado de forma inmediata.

In [ ]:
import os
import sqlite3
import pandas as pd
import numpy as np

In [ ]:
# Definir ruta de la nueva base de datos y recrearla
database = '/content/drive/MyDrive/notebooks_introduccion/datos/estaciones_meteo.sqlite'
if os.path.exists(database):
    os.remove(database)

conn = sqlite3.connect(database)
cursor = conn.cursor()

# 1. Crear tabla info_estaciones
cursor.execute('''
CREATE TABLE info_estaciones (
    id_estacion INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre TEXT,
    provincia TEXT,
    latitud REAL,
    longitud REAL
)
''')

# 2. Crear tabla mediciones
cursor.execute('''
CREATE TABLE mediciones (
    id_medicion INTEGER PRIMARY KEY AUTOINCREMENT,
    temp REAL,
    presion REAL,
    direccion_viento TEXT,
    velocidad_viento REAL,
    precipitacion REAL,
    estacion TEXT,
    hora TEXT,
    fecha TEXT
)
''')

# Insertar datos sintéticos de estaciones para varias provincias
estaciones_data = [
    ('Estacion Aeroparque', 'Buenos Aires', -34.558, -58.416),
    ('Estacion Ezeiza', 'Buenos Aires', -34.822, -58.535),
    ('Estacion Cordoba Centro', 'Córdoba', -31.413, -64.181),
    ('Estacion Mendoza Plaza', 'Mendoza', -32.889, -68.845),
    ('Estacion Rosario Sur', 'Santa Fe', -32.946, -60.639)
]
cursor.executemany(
    "INSERT INTO info_estaciones (nombre, provincia, latitud, longitud) VALUES (?, ?, ?, ?)",
    estaciones_data
)

# Insertar datos sintéticos de series de tiempo para mediciones
fechas = pd.date_range(start="2023-10-01", end="2023-10-05", freq="D").strftime("%Y-%m-%d").tolist()
horas = ["08:00", "14:00", "20:00"]
estaciones_nombres = [e[0] for e in estaciones_data]

mediciones_data = []
np.random.seed(42)
for est in estaciones_nombres:
    for fecha in fechas:
        for hora in horas:
            temp = round(np.random.uniform(10.0, 35.0), 1)
            presion = round(np.random.uniform(980.0, 1025.0), 1)
            dir_viento = np.random.choice(['N', 'S', 'E', 'O', 'NE', 'NO', 'SE', 'SO'])
            vel_viento = round(np.random.uniform(5.0, 45.0), 1)
            precip = round(np.random.exponential(1.5) if np.random.rand() > 0.7 else 0.0, 1)
            mediciones_data.append((temp, presion, dir_viento, vel_viento, precip, est, hora, fecha))

cursor.executemany(
    "INSERT INTO mediciones (temp, presion, direccion_viento, velocidad_viento, precipitacion, estacion, hora, fecha) VALUES (?, ?, ?, ?, ?, ?, ?, ?)",
    mediciones_data
)

conn.commit()
conn.close()
print("Base de datos recreada, tablas configuradas e insertados datos sintéticos de prueba exitosamente.")

Base de datos recreada, tablas configuradas e insertados datos sintéticos de prueba exitosamente.


## Práctico: Consulta y Series Temporales

Ahora que la base de datos cuenta con la estructura correcta y datos sintéticos de diferentes provincias, vamos a resolver los ejercicios.

### Práctico 7.1: Estaciones en una Provincia Específica

Escribe y ejecuta una consulta SQL para seleccionar todas las estaciones meteorológicas ubicadas en la provincia de `'Buenos Aires'` desde la tabla `info_estaciones`. Carga los resultados en un DataFrame llamado `df_buenos_aires`.

In [ ]:
conn = sqlite3.connect(database)

# Consulta SQL filtrando por la provincia especificada
consulta_sql = "SELECT * FROM info_estaciones WHERE provincia = 'Buenos Aires'"

df_buenos_aires = pd.read_sql_query(consulta_sql, conn)
conn.close()

display(df_buenos_aires)

,id_estacion,nombre,provincia,latitud,longitud
0,1,Estacion Aeroparque,Buenos Aires,-34.558,-58.416
1,2,Estacion Ezeiza,Buenos Aires,-34.822,-58.535


In [ ]:
# CELDA DE VERIFICACIÓN - PRÁCTICO 7.1
try:
    assert 'df_buenos_aires' in globals(), 'El DataFrame df_buenos_aires no está definido.'
    assert isinstance(df_buenos_aires, pd.DataFrame), 'df_buenos_aires debe ser un DataFrame de Pandas.'
    assert not df_buenos_aires.empty, 'El DataFrame está vacío. Verifica tu consulta SQL.'
    assert (df_buenos_aires['provincia'] == 'Buenos Aires').all(), 'Hay estaciones que no corresponden a Buenos Aires.'
    print('Práctico 7.1 superado y verificado con éxito. 🗄️')
except AssertionError as e:
    print(f'Error de validación: {e}')

Práctico 7.1 superado y verificado con éxito. 🗄️


### Práctico 7.2: Serie de Tiempo de una Estación

Obtén la serie temporal de temperatura e historial de precipitaciones para la `'Estacion Cordoba Centro'` ordenada de forma cronológica por fecha y hora.

In [ ]:
conn = sqlite3.connect(database)

consulta_serie = """
SELECT fecha, hora, temp, precipitacion
FROM mediciones
WHERE estacion = 'Estacion Cordoba Centro'
ORDER BY fecha ASC, hora ASC
"""

df_serie_tiempo = pd.read_sql_query(consulta_serie, conn)
conn.close()

display(df_serie_tiempo)

,fecha,hora,temp,precipitacion
0,2023-10-01,08:00,10.9,0.0
1,2023-10-01,14:00,18.6,0.0
2,2023-10-01,20:00,34.6,0.0
3,2023-10-02,08:00,16.1,0.0
4,2023-10-02,14:00,23.4,0.0
5,2023-10-02,20:00,27.4,0.0
6,2023-10-03,08:00,22.8,0.8
7,2023-10-03,14:00,22.9,0.0
8,2023-10-03,20:00,33.1,0.0
9,2023-10-04,08:00,34.6,0.0


In [ ]:
# CELDA DE VERIFICACIÓN - PRÁCTICO 7.2
try:
    assert 'df_serie_tiempo' in globals(), 'El DataFrame df_serie_tiempo no está definido.'
    assert isinstance(df_serie_tiempo, pd.DataFrame), 'df_serie_tiempo debe ser un DataFrame de Pandas.'
    assert not df_serie_tiempo.empty, 'El DataFrame está vacío. Verifica tu consulta SQL.'
    columnas_esperadas = ['fecha', 'hora', 'temp', 'precipitacion']
    for col in columnas_esperadas:
        assert col in df_serie_tiempo.columns, f'Falta la columna esperada: {col}'
    # Verificar orden cronológico
    ordenado = df_serie_tiempo.equals(df_serie_tiempo.sort_values(by=['fecha', 'hora']))
    assert ordenado, 'Los datos no están ordenados cronológicamente por fecha y hora.'
    print('Práctico 7.2 superado y verificado con éxito. 📈')
except AssertionError as e:
    print(f'Error de validación: {e}')

Práctico 7.2 superado y verificado con éxito. 📈


In [13]:
from sqlalchemy import create_engine

# Crear el motor de conexión con SQLAlchemy
# Usamos la ruta de la base de datos definida previamente en la variable `database`
engine = create_engine(f'sqlite:///{database}')

# 1. Práctico 7.1 utilizando SQLAlchemy
consulta_sql = "SELECT * FROM info_estaciones WHERE provincia = 'Buenos Aires'"
df_buenos_aires = pd.read_sql_query(consulta_sql, con=engine)

print("--- Práctico 7.1 (SQLAlchemy) ---")
display(df_buenos_aires)

# 2. Práctico 7.2 utilizando SQLAlchemy
consulta_serie = """
SELECT fecha, hora, temp, precipitacion
FROM mediciones
WHERE estacion = 'Estacion Cordoba Centro'
ORDER BY fecha ASC, hora ASC
"""
df_serie_tiempo = pd.read_sql_query(consulta_serie, con=engine)

print("\n--- Práctico 7.2 (SQLAlchemy) ---")
display(df_serie_tiempo)

# Cerrar o disponer el engine de SQLAlchemy
engine.dispose()

--- Práctico 7.1 (SQLAlchemy) ---


,id_estacion,nombre,provincia,latitud,longitud
0,1,Estacion Aeroparque,Buenos Aires,-34.558,-58.416
1,2,Estacion Ezeiza,Buenos Aires,-34.822,-58.535



--- Práctico 7.2 (SQLAlchemy) ---


,fecha,hora,temp,precipitacion
0,2023-10-01,08:00,10.9,0.0
1,2023-10-01,14:00,18.6,0.0
2,2023-10-01,20:00,34.6,0.0
3,2023-10-02,08:00,16.1,0.0
4,2023-10-02,14:00,23.4,0.0
5,2023-10-02,20:00,27.4,0.0
6,2023-10-03,08:00,22.8,0.8
7,2023-10-03,14:00,22.9,0.0
8,2023-10-03,20:00,33.1,0.0
9,2023-10-04,08:00,34.6,0.0


### Representación de la Base de Datos usando el ORM de SQLAlchemy

El mapeo objeto-relacional (ORM) asocia clases de Python con tablas de bases de datos. Primero, definimos nuestra base declarativa y mapeamos las tablas `info_estaciones` y `mediciones` a clases de Python.

In [14]:
from sqlalchemy import create_engine, Column, Integer, String, Float
from sqlalchemy.orm import declarative_base, sessionmaker

# Creamos la base declarativa
Base = declarative_base()

# Definimos el modelo para la tabla 'info_estaciones'
class InfoEstacion(Base):
    __tablename__ = 'info_estaciones'

    id_estacion = Column(Integer, primary_key=True, autoincrement=True)
    nombre = Column(String)
    provincia = Column(String)
    latitud = Column(Float)
    longitud = Column(Float)

# Definimos el modelo para la tabla 'mediciones'
class Medicion(Base):
    __tablename__ = 'mediciones'

    id_medicion = Column(Integer, primary_key=True, autoincrement=True)
    temp = Column(Float)
    presion = Column(Float)
    direccion_viento = Column(String)
    velocidad_viento = Column(Float)
    precipitacion = Column(Float)
    estacion = Column(String)
    hora = Column(String)
    fecha = Column(String)

# Creamos el motor y el constructor de sesiones
engine = create_engine(f'sqlite:///{database}')
Session = sessionmaker(bind=engine)
session = Session()

### Práctico 7.1 Revisitado con ORM (Sin SQL)

Ahora realizaremos la consulta para obtener las estaciones de la provincia de `'Buenos Aires'` utilizando la sesión de SQLAlchemy y el método `.filter()`, convirtiendo luego los objetos mapeados directamente a un DataFrame de Pandas.

In [15]:
# Realizar la consulta usando la API del ORM
query_estaciones = session.query(InfoEstacion).filter(InfoEstacion.provincia == 'Buenos Aires').all()

# Convertir la lista de objetos de Python a un DataFrame
estaciones_dict = [
    {
        'id_estacion': est.id_estacion,
        'nombre': est.nombre,
        'provincia': est.provincia,
        'latitud': est.latitud,
        'longitud': est.longitud
    }
    for est in query_estaciones
]
df_buenos_aires_orm = pd.DataFrame(estaciones_dict)

print("--- Práctico 7.1 (SQLAlchemy ORM) ---")
display(df_buenos_aires_orm)

--- Práctico 7.1 (SQLAlchemy ORM) ---


,id_estacion,nombre,provincia,latitud,longitud
0,1,Estacion Aeroparque,Buenos Aires,-34.558,-58.416
1,2,Estacion Ezeiza,Buenos Aires,-34.822,-58.535


### Práctico 7.2 Revisitado con ORM (Sin SQL)

Del mismo modo, obtenemos la serie temporal para la `'Estacion Cordoba Centro'` filtrando por estación y ordenando por `fecha` y `hora` de manera cronológica usando `.order_by()`.

In [16]:
# Realizar la consulta con filtros y ordenamiento nativos del ORM
query_mediciones = (
    session.query(Medicion.fecha, Medicion.hora, Medicion.temp, Medicion.precipitacion)
    .filter(Medicion.estacion == 'Estacion Cordoba Centro')
    .order_by(Medicion.fecha.asc(), Medicion.hora.asc())
    .all()
)

# Cargar los atributos seleccionados directamente en un DataFrame de Pandas
df_serie_tiempo_orm = pd.DataFrame(query_mediciones, columns=['fecha', 'hora', 'temp', 'precipitacion'])

print("--- Práctico 7.2 (SQLAlchemy ORM) ---")
display(df_serie_tiempo_orm)

# Cerramos la sesión al finalizar
session.close()

--- Práctico 7.2 (SQLAlchemy ORM) ---


,fecha,hora,temp,precipitacion
0,2023-10-01,08:00,10.9,0.0
1,2023-10-01,14:00,18.6,0.0
2,2023-10-01,20:00,34.6,0.0
3,2023-10-02,08:00,16.1,0.0
4,2023-10-02,14:00,23.4,0.0
5,2023-10-02,20:00,27.4,0.0
6,2023-10-03,08:00,22.8,0.8
7,2023-10-03,14:00,22.9,0.0
8,2023-10-03,20:00,33.1,0.0
9,2023-10-04,08:00,34.6,0.0
